In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from enum import Enum
import pygame
import pong_engine
from notebooks.models.neural_network import learning_rate

np.set_printoptions(precision=3,suppress=True)

In [100]:
#Creating a enum for loss functions
class LossFunction(Enum):
    MSE = 1
    BCE = 2
    CCE = 3
    REINFORCE = 4

In [ ]:
#Creating a enum for weights initialization
class WeightInitialization(Enum):
    Random = 1
    Xavier = 2
    He = 3

In [ ]:
class Layer:
    def __init__(self):
        return

    def forward_pass(self,x):
        return

    def backward_pass(self, derivative):
        return

    def update(self, derivative):
        return

In [ ]:
class Sigmoid(Layer):
    def __init__(self):
        super().__init__()
        self.sigmoid_output = None

    def forward_pass(self,x):
        self.sigmoid_output = 1 / (1 + np.exp(-x))
        return self.sigmoid_output

    def backward_pass(self, derivative):
        return derivative * self.sigmoid_output*(1 - self.sigmoid_output)

In [ ]:
#Standard relu as a layer, does not need to know params, will use X to map out gradient then apply it to incoming derivatives
class Relu(Layer):
    def __init__(self):
        super().__init__()
        self.x = None
        self.gradient = None

    def forward_pass(self,x):
        self.gradient = np.where(x > 0, 1, 0)
        self.x = x
        return self.x * self.gradient

    def backward_pass(self, derivative):
        return derivative * self.gradient

In [ ]:
#Class for leaky ReLU, does not need to know params, will use X to map out gradient then apply it to incoming derivatives
class LeakyRelu(Layer):
    def __init__(self, alpha=0.01):
        super().__init__()
        self.x = None
        self.gradient = None
        self.alpha = alpha
    def forward_pass(self,x):
        self.gradient = np.where(x > 0, 1, self.alpha)
        self.x = x
        return self.x * self.gradient
    def backward_pass(self, derivative):
        return derivative * self.gradient

In [ ]:
#Class for softmax layer, does not need to know params, will use X to map out gradient then apply it to incoming derivatives
class Softmax(Layer):
    def __init__(self):
        super().__init__()
        self.probabilities = None

    def forward_pass(self,x):
        temp = np.exp(x)
        denominator = np.sum(temp, axis=0, keepdims=True)
        self.probabilities = temp / denominator
        return self.probabilities

    def backward_pass(self, derivative):
        sum = np.sum(derivative * self.probabilities, axis=0, keepdims=True)
        return self.probabilities * (derivative - sum)

    def update(self, derivative):
        return

In [ ]:
class Linear(Layer):
    def __init__(self,input_dim,output_dim,learning_rate=0.001,weight_init=WeightInitialization.Random,weights=None,biases=None):
        super().__init__()
        if weights is None:
            match weight_init:
                case WeightInitialization.Random:
                    weights = np.random.uniform(-1,1,size=(output_dim,input_dim))
                case WeightInitialization.Xavier:
                    weights = np.random.uniform(-np.sqrt(6.0/(input_dim + output_dim)),np.sqrt(6.0/(input_dim + output_dim)),size=(output_dim,input_dim))
                case WeightInitialization.He:
                    weights = np.random.normal(0, np.sqrt(2.0/input_dim), size=(output_dim,input_dim))
        if biases is None:
            biases = np.full(shape=(output_dim,1),fill_value=0,dtype=float)
        self.learning_rate = learning_rate
        self.x = None
        self.weights = weights
        self.biases = biases
        self.input_dim = input_dim
        self.output_dim = output_dim

    def forward_pass(self,x):
        self.x = x
        return self.weights @ x + self.biases

    def backward_pass(self, derivative):
        return self.weights.T @ derivative

    def update(self, derivative):
        weight_gradient = derivative @ self.x.T
        bias_gradient = np.mean(derivative, axis=1,keepdims=True)

        self.weights -= self.learning_rate * weight_gradient
        self.biases -= self.learning_rate * bias_gradient

In [101]:
class Model:
    def __init__(self, layers):
        self.layers = layers

    def calc_loss(self,y_hat,y,loss_function=LossFunction.MSE):
        match loss_function:
            case LossFunction.MSE:
                return y_hat - y
            case LossFunction.BCE:
                return (y_hat-y)/(y_hat*(1-y_hat))
            case LossFunction.CCE:
                return y*np.log(y_hat)
            case LossFunction.REINFORCE:
                # Create a one-hot encoded array for the chosen action
                choice = np.array(shape=(y_hat.shape))
                choice[y_hat.index(max(y_hat))] = 1
                return y*np.log(choice-y_hat)

    def calc_cost(self,y_hat,y,loss_function=LossFunction.MSE):
        match loss_function:
            case LossFunction.MSE:
                return np.mean((y_hat-y)**2)/2
            case LossFunction.BCE:
                return -np.mean(y*np.log(y_hat) + (1-y)*np.log(1-y_hat))
            case LossFunction.CCE:
                return -np.mean(np.sum(y*np.log(y_hat),axis=0,keepdims=True))
            case LossFunction.REINFORCE:
                return np.mean(np.ones(shape=(y_hat.shape[0]))-np.max(y_hat,axis=0,keepdims=True))

    def forward_pass(self,x):
        inp = x
        for i in self.layers:
            inp = i.forward_pass(inp)
        return inp

    def backward_pass(self, derivative):
        gradient = derivative
        for i in reversed(self.layers):
            temp_gradient = i.backward_pass(gradient)
            i.update(gradient)
            gradient = temp_gradient

    #Input and output is given in as: batch,input/output
    def train(self,x,y,iterations=10000,cutoff = 0.001,loss_function=LossFunction.MSE):
        cost_history = []
        for i in range(iterations):
            prediction = self.forward_pass(x)
            loss = self.calc_loss(prediction, y, loss_function)
            cost_history.append(self.calc_cost(prediction, y, loss_function))
            self.backward_pass(loss)
            if len(cost_history) < 2:
                continue
            if abs((cost_history[-1]-cost_history[-2])/cost_history[-1]) < cutoff:
                break
        return cost_history

    #Input and output is given in as: batch,input/output
    def advanced_train(self, x, y, epoch, batch_size, iterations=100000, cutoff = 0.001,loss_function=LossFunction.MSE):
        cost_history = []
        counter = 0
        for i in range(epoch):
            batches = self.create_batches(x,y,batch_size)
            for batch in batches:
                batch_x = batch[0]
                batch_y = batch[1]
                prediction = self.forward_pass(batch_x)
                loss = self.calc_loss(prediction, batch_y,loss_function)
                cost_history.append(self.calc_cost(prediction, batch_y,loss_function))
                self.backward_pass(loss)
                if len(cost_history) < 2:
                    continue
                if abs((cost_history[-1]-cost_history[-2])/cost_history[-1]) < cutoff:
                    return cost_history
                if counter == iterations:
                    break
                counter += 1
        return cost_history
        return

    def create_batches(self,x,y,batch_size):
        batches = []
        if batch_size > len(x):
            raise Exception("Batch size cannot be larger than the dataset size.")
        indexes = np.arange(len(x))
        np.random.shuffle(indexes)
        x_shuffled = x[indexes]
        y_shuffled = y[indexes]
        for i in range(0,x.shape[0],batch_size):
            batch_x = x_shuffled[i:i+batch_size].T
            batch_y = y_shuffled[i:i+batch_size].T
            batches.append((batch_x,batch_y))
        return batches

Model

In [ ]:
learning_rate = 0.001;

model = Model([
    Linear(8,16,learning_rate),
    Relu(),
    Linear(16,16,learning_rate),
    Relu(),
    Linear(16,3,learning_rate),
    Softmax()
])

def state_to_np(game_state,field_width,field_height,NORMALIZATION_FACTOR):
    return np.array([
        float(field_width) / NORMALIZATION_FACTOR,
        float(field_height) / NORMALIZATION_FACTOR,
        float(game_state.ball_x) / NORMALIZATION_FACTOR * 10,
        float(game_state.ball_y) / NORMALIZATION_FACTOR * 10,
        float(game_state.player_one_x) / NORMALIZATION_FACTOR,
        float(game_state.player_one_y) / NORMALIZATION_FACTOR,
        float(game_state.player_two_x) / NORMALIZATION_FACTOR,
        float(game_state.player_two_y) / NORMALIZATION_FACTOR,
    ], dtype=float)

def model_wrapper_forward_pass(field_width, field_height, game_state, prob=False):
    global model

    # The exported C++ Game object does not expose __dict__ or live state attributes.
    # We only have access to the last StepResult returned by game.step(...). Use that.
    NORMALIZATION_FACTOR = 100.0
    state_vector = state_to_np(game_state, field_width, field_height, NORMALIZATION_FACTOR)

    prediction = model.forward_pass(state_vector.reshape(-1, 1))
    action_index = int(np.argmax(prediction, axis=0)[0])
    if prob:
        return prediction
    # model output shape is 3x1 for [UP, NONE, DOWN]
    action_map = {
        0: pong_engine.Move.UP,
        1: pong_engine.Move.NONE,
        2: pong_engine.Move.DOWN,
    }
    return action_map.get(action_index, pong_engine.Move.NONE)


Game code

In [ ]:
FIELD_WIDTH = 640
FIELD_HEIGHT = 480
BALL_RADIUS = 10
PLAYER_WIDTH = 10
PLAYER_HEIGHT = 100
PLAYER_OFFSET = 80
PLAYER_ACCEL = 2000.0
PLAYER_TOP_SPEED = 500.0
BALL_ACCEL = 1.05
BALL_VEL = 170.0

game = pong_engine.Game(
    ball_radius=BALL_RADIUS,
    player_width=PLAYER_WIDTH,
    player_height=PLAYER_HEIGHT,
    field_width=FIELD_WIDTH,
    field_height=FIELD_HEIGHT,
    player_offset=PLAYER_OFFSET,
    player_accel=PLAYER_ACCEL,
    player_top_speed=PLAYER_TOP_SPEED,
    ball_accel=BALL_ACCEL,
    ball_vel=BALL_VEL,
)

In [94]:
events = []

pygame.init()
screen = pygame.display.set_mode((FIELD_WIDTH, FIELD_HEIGHT))
pygame.display.set_caption('Pong')
clock = pygame.time.Clock()

def get_move_from_keys(up_key, down_key):
    if up_key and not down_key:
        return pong_engine.Move.UP
    if down_key and not up_key:
        return pong_engine.Move.DOWN
    return pong_engine.Move.NONE

last_state = None
running = True
while running:
    delta_time = clock.tick(60) / 1000.0

    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False

    if last_state is None:
        last_state = game.step(pong_engine.Move.NONE, pong_engine.Move.NONE, delta_time)

    pressed = pygame.key.get_pressed()
    left_move = model_wrapper_forward_pass(FIELD_WIDTH, FIELD_HEIGHT, last_state)
    right_move = get_move_from_keys(pressed[pygame.K_UP], pressed[pygame.K_DOWN])
    events.append((last_state, left_move, delta_time))
    result = game.step(left_move, right_move, delta_time)
    last_state = result

    screen.fill((0, 0, 0))

    pygame.draw.rect(screen, (255, 255, 255), pygame.Rect(0, 0, FIELD_WIDTH, FIELD_HEIGHT), 2)
    pygame.draw.line(screen, (255, 255, 255), (FIELD_WIDTH // 2, 0), (FIELD_WIDTH // 2, FIELD_HEIGHT), 2)

    left_paddle = pygame.Rect(
        int(result.player_one_x),
        int(result.player_one_y),
        PLAYER_WIDTH,
        PLAYER_HEIGHT,
    )
    right_paddle = pygame.Rect(
        int(result.player_two_x),
        int(result.player_two_y),
        PLAYER_WIDTH,
        PLAYER_HEIGHT,
    )
    pygame.draw.rect(screen, (255, 255, 255), left_paddle)
    pygame.draw.rect(screen, (255, 255, 255), right_paddle)
    pygame.draw.circle(screen, (255, 255, 255), (int(result.ball_x), int(result.ball_y)), BALL_RADIUS)
    pygame.display.flip()

    if result.game_over:
        print(f'Game over! Player {result.winner} wins.')
        running = False
events.append((last_state,0))
pygame.quit()
print(events)

Game over! Player 2 wins.
[(<pong_engine.StepResult object at 0x7d6607dadcb0>, <Move.UP: 1>, 0.016), (<pong_engine.StepResult object at 0x7d6607ddd870>, <Move.UP: 1>, 0.016), (<pong_engine.StepResult object at 0x7d660a9f39b0>, <Move.UP: 1>, 0.016), (<pong_engine.StepResult object at 0x7d6607dae970>, <Move.UP: 1>, 0.016), (<pong_engine.StepResult object at 0x7d6607dadbf0>, <Move.UP: 1>, 0.016), (<pong_engine.StepResult object at 0x7d6607dadb70>, <Move.UP: 1>, 0.016), (<pong_engine.StepResult object at 0x7d6607dadbb0>, <Move.UP: 1>, 0.017), (<pong_engine.StepResult object at 0x7d6607dadaf0>, <Move.UP: 1>, 0.016), (<pong_engine.StepResult object at 0x7d6607dadb30>, <Move.UP: 1>, 0.016), (<pong_engine.StepResult object at 0x7d6607dadc30>, <Move.UP: 1>, 0.016), (<pong_engine.StepResult object at 0x7d6607dada70>, <Move.UP: 1>, 0.016), (<pong_engine.StepResult object at 0x7d6607dadab0>, <Move.UP: 1>, 0.016), (<pong_engine.StepResult object at 0x7d6607dad9f0>, <Move.UP: 1>, 0.016), (<pong_engi

In [92]:
def assign_reward(position,reward,loss,rewards):
    for i in range(position,-1,-1):
        rewards[i] += reward
        reward *= loss

In [95]:
reward = np.full(shape=(100), fill_value=0.1)
if (events[-1][0].winner == 1):
    assign_reward(len(reward)-1,10,0.9,reward)
elif (events[-1][0].winner == 2):
    assign_reward(len(reward)-1,-10,0.9,reward)


-1
